In [ ]:
# --- 1. PyTorch/XLA (install only if the TPU image lacks it) ---
import importlib.util, subprocess, sys
if importlib.util.find_spec("torch_xla") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.9.0", "torch_xla[tpu]==2.9.0",
                    "-f", "https://storage.googleapis.com/libtpu-releases/index.html"], check=True)
print(subprocess.run([sys.executable, "-c", "import torch, torch_xla; print('torch', torch.__version__, '| torch_xla', torch_xla.__version__)"],
                     capture_output=True, text=True).stdout)


In [ ]:
# --- 2. Vishalam model source (embedded) + benchmark script ---
import os
os.makedirs(os.path.dirname('/kaggle/working/vishalam_src/vishalam/__init__.py'), exist_ok=True)
open('/kaggle/working/vishalam_src/vishalam/__init__.py', 'w').write('')
os.makedirs(os.path.dirname('/kaggle/working/vishalam_src/vishalam/model/__init__.py'), exist_ok=True)
open('/kaggle/working/vishalam_src/vishalam/model/__init__.py', 'w').write('')
os.makedirs(os.path.dirname('/kaggle/working/vishalam_src/vishalam/model/config.py'), exist_ok=True)
open('/kaggle/working/vishalam_src/vishalam/model/config.py', 'w').write('from __future__ import annotations\n\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path\n\nimport yaml\n\n\n@dataclass\nclass ModelConfig:\n    vocab_size: int = 50304\n    d_model: int = 1024\n    n_layers: int = 24\n    n_heads: int = 16\n    ffn_dim: int = 2816\n    max_seq_len: int = 4096\n    rope_theta: float = 10000.0\n    norm_eps: float = 1e-5\n    tie_embeddings: bool = True\n\n    @property\n    def head_dim(self) -> int:\n        return self.d_model // self.n_heads\n\n    def to_dict(self) -> dict:\n        return asdict(self)\n\n\ndef load_yaml(path: str | Path) -> dict:\n    return yaml.safe_load(Path(path).read_text(encoding="utf-8"))\n\n\ndef model_config_from_yaml(path: str | Path) -> ModelConfig:\n    return ModelConfig(**load_yaml(path)["model"])\n')
os.makedirs(os.path.dirname('/kaggle/working/vishalam_src/vishalam/model/model.py'), exist_ok=True)
open('/kaggle/working/vishalam_src/vishalam/model/model.py', 'w').write('"""Vishalam decoder-only transformer, written from scratch.\n\nLlama-style: pre-norm RMSNorm, rotary position embeddings (RoPE), SwiGLU feed-forward,\nno biases, tied input/output embeddings, PyTorch SDPA attention, optional KV cache for\nfast generation and optional activation checkpointing for training memory.\n"""\nfrom __future__ import annotations\n\nimport math\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torch.utils.checkpoint import checkpoint\n\nfrom .config import ModelConfig\n\n\nclass RMSNorm(nn.Module):\n    def __init__(self, dim: int, eps: float):\n        super().__init__()\n        self.eps = eps\n        self.weight = nn.Parameter(torch.ones(dim))\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        xf = x.float()\n        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)\n        return (xf * self.weight.float()).to(x.dtype)\n\n\ndef rope_tables(head_dim: int, max_len: int, theta: float) -> tuple[torch.Tensor, torch.Tensor]:\n    inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim))\n    t = torch.arange(max_len, dtype=torch.float32)\n    freqs = torch.outer(t, inv_freq)  # (T, hd/2)\n    return freqs.cos(), freqs.sin()\n\n\ndef apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:\n    # x: (B, H, T, hd); cos/sin: (T, hd/2). Rotate pairs (first half, second half).\n    x1, x2 = x.float().chunk(2, dim=-1)\n    cos, sin = cos[None, None], sin[None, None]\n    out = torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)\n    return out.to(x.dtype)\n\n\nclass KVCache:\n    """Preallocated per-layer key/value cache for incremental decoding."""\n\n    def __init__(self, cfg: ModelConfig, batch: int, max_len: int, device, dtype):\n        shape = (batch, cfg.n_heads, max_len, cfg.head_dim)\n        self.k = [torch.zeros(shape, device=device, dtype=dtype) for _ in range(cfg.n_layers)]\n        self.v = [torch.zeros(shape, device=device, dtype=dtype) for _ in range(cfg.n_layers)]\n        self.max_len = max_len\n        self.pos = 0  # number of positions already filled\n\n    def truncate(self, pos: int) -> None:\n        self.pos = pos\n\n\nclass Attention(nn.Module):\n    def __init__(self, cfg: ModelConfig):\n        super().__init__()\n        self.n_heads, self.head_dim = cfg.n_heads, cfg.head_dim\n        self.qkv = nn.Linear(cfg.d_model, 3 * cfg.d_model, bias=False)\n        self.proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)\n\n    def forward(self, x, cos, sin, cache: KVCache | None = None, layer: int = 0):\n        B, T, C = x.shape\n        q, k, v = self.qkv(x).split(C, dim=-1)\n        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)\n        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)\n        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)\n        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)\n\n        if cache is None:\n            y = F.scaled_dot_product_attention(q, k, v, is_causal=True)\n        else:\n            start = cache.pos\n            cache.k[layer][:, :, start : start + T] = k\n            cache.v[layer][:, :, start : start + T] = v\n            k_all = cache.k[layer][:, :, : start + T]\n            v_all = cache.v[layer][:, :, : start + T]\n            if T == 1:\n                y = F.scaled_dot_product_attention(q, k_all, v_all)\n            else:\n                # query i (absolute pos start+i) may attend keys 0..start+i\n                qpos = torch.arange(start, start + T, device=x.device)[:, None]\n                kpos = torch.arange(start + T, device=x.device)[None, :]\n                y = F.scaled_dot_product_attention(q, k_all, v_all, attn_mask=kpos <= qpos)\n        y = y.transpose(1, 2).contiguous().view(B, T, C)\n        return self.proj(y)\n\n\nclass SwiGLU(nn.Module):\n    def __init__(self, cfg: ModelConfig):\n        super().__init__()\n        self.w_gate = nn.Linear(cfg.d_model, cfg.ffn_dim, bias=False)\n        self.w_up = nn.Linear(cfg.d_model, cfg.ffn_dim, bias=False)\n        self.w_down = nn.Linear(cfg.ffn_dim, cfg.d_model, bias=False)\n\n    def forward(self, x):\n        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))\n\n\nclass Block(nn.Module):\n    def __init__(self, cfg: ModelConfig):\n        super().__init__()\n        self.norm1 = RMSNorm(cfg.d_model, cfg.norm_eps)\n        self.attn = Attention(cfg)\n        self.norm2 = RMSNorm(cfg.d_model, cfg.norm_eps)\n        self.mlp = SwiGLU(cfg)\n\n    def forward(self, x, cos, sin, cache=None, layer=0):\n        x = x + self.attn(self.norm1(x), cos, sin, cache, layer)\n        return x + self.mlp(self.norm2(x))\n\n\nclass Vishalam(nn.Module):\n    def __init__(self, cfg: ModelConfig):\n        super().__init__()\n        self.cfg = cfg\n        self.embed = nn.Embedding(cfg.vocab_size, cfg.d_model)\n        self.blocks = nn.ModuleList(Block(cfg) for _ in range(cfg.n_layers))\n        self.norm = RMSNorm(cfg.d_model, cfg.norm_eps)\n        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)\n        if cfg.tie_embeddings:\n            self.lm_head.weight = self.embed.weight\n        cos, sin = rope_tables(cfg.head_dim, cfg.max_seq_len, cfg.rope_theta)\n        self.register_buffer("rope_cos", cos, persistent=False)\n        self.register_buffer("rope_sin", sin, persistent=False)\n        self.grad_checkpointing = False\n        self.apply(self._init)\n        # scale residual-branch output projections (GPT-2 style)\n        for name, p in self.named_parameters():\n            if name.endswith("attn.proj.weight") or name.endswith("mlp.w_down.weight"):\n                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * cfg.n_layers))\n\n    @staticmethod\n    def _init(m: nn.Module) -> None:\n        if isinstance(m, (nn.Linear, nn.Embedding)):\n            nn.init.normal_(m.weight, mean=0.0, std=0.02)\n\n    def _apply(self, fn, *args, **kwargs):\n        super()._apply(fn, *args, **kwargs)\n        # PyTorch/XLA rebuilds every parameter on .to(device), which would untie the output head from\n        # the embedding (a different model, and checkpoints that no longer load across GPU/TPU)\n        if self.cfg.tie_embeddings:\n            self.lm_head.weight = self.embed.weight\n        return self\n\n    def num_params(self) -> int:\n        return sum(p.numel() for p in self.parameters())\n\n    def forward(\n        self,\n        idx: torch.Tensor,\n        targets: torch.Tensor | None = None,\n        loss_mask: torch.Tensor | None = None,\n        cache: KVCache | None = None,\n        z_loss: float = 0.0,\n        return_logits: bool = False,\n    ):\n        """idx: (B, T) token ids. targets: (B, T) next-token ids (or None).\n        loss_mask: (B, T) 1 where the target should count toward the loss (e.g. agent tokens only).\n        Returns (logits, loss). With targets, logits are None unless return_logits=True (training uses a\n        memory-saving chunked loss). Without targets (inference), only the last position\'s logits are\n        computed.\n        """\n        B, T = idx.shape\n        start = cache.pos if cache is not None else 0\n        assert start + T <= self.cfg.max_seq_len, "sequence longer than max_seq_len"\n        cos = self.rope_cos[start : start + T]\n        sin = self.rope_sin[start : start + T]\n        x = self.embed(idx)\n        for i, blk in enumerate(self.blocks):\n            if self.grad_checkpointing and self.training and cache is None:\n                x = checkpoint(blk, x, cos, sin, use_reentrant=False)\n            else:\n                x = blk(x, cos, sin, cache, i)\n        x = self.norm(x)\n        if cache is not None:\n            cache.pos = start + T\n\n        if targets is None:\n            return self.lm_head(x[:, -1:]).float(), None\n\n        flat_targets = targets.reshape(-1)\n        if loss_mask is None:\n            weights = torch.ones(flat_targets.shape, device=x.device, dtype=torch.float32)\n        else:\n            weights = loss_mask.reshape(-1).to(torch.float32)\n        denom = weights.sum().clamp(min=1.0)\n\n        if return_logits:\n            logits = self.lm_head(x).float()\n            ce_sum, z_sum = _loss_terms(logits.view(-1, logits.size(-1)), flat_targets, weights, z_loss)\n        else:\n            # Chunked loss: never materialise (B*T, vocab) fp32 logits at once. Each chunk\'s logits are\n            # recomputed in backward (checkpoint), so peak memory is one chunk instead of the whole batch.\n            # Chunks run along the sequence (all rows of the batch together): with the batch split\n            # across TPU chips, every chip then only touches its own rows.\n            logits = None\n            ce_sum = z_sum = torch.zeros((), device=x.device, dtype=torch.float32)\n            w = weights.view(B, T)\n            span = max(1, LOSS_CHUNK // B)\n            for s in range(0, T, span):\n                part = (x[:, s : s + span].reshape(-1, x.size(-1)), targets[:, s : s + span].reshape(-1),\n                        w[:, s : s + span].reshape(-1), z_loss)\n                if torch.is_grad_enabled():\n                    c, z = checkpoint(self._chunk_loss, *part, use_reentrant=False)\n                else:\n                    c, z = self._chunk_loss(*part)\n                ce_sum, z_sum = ce_sum + c, z_sum + z\n        loss = ce_sum / denom\n        if z_loss > 0:\n            loss = loss + z_loss * z_sum / denom\n        return logits, loss\n\n    def _chunk_loss(self, h, t, w, z_loss: float):\n        return _loss_terms(self.lm_head(h).float(), t, w, z_loss)\n\n\nLOSS_CHUNK = 4096  # tokens per loss chunk (~0.8 GB of fp32 logits for a 50k vocab)\n\n\ndef _loss_terms(flat_logits, targets, weights, z_loss: float):\n    ce = F.cross_entropy(flat_logits, targets, reduction="none")\n    ce_sum = (ce * weights).sum()\n    if z_loss > 0:\n        lse = torch.logsumexp(flat_logits, dim=-1)\n        z_sum = ((lse**2) * weights).sum()\n    else:\n        z_sum = torch.zeros((), device=flat_logits.device, dtype=torch.float32)\n    return ce_sum, z_sum\n')
os.makedirs(os.path.dirname('/kaggle/working/vishalam_src/vishalam/train/__init__.py'), exist_ok=True)
open('/kaggle/working/vishalam_src/vishalam/train/__init__.py', 'w').write('')
os.makedirs(os.path.dirname('/kaggle/working/vishalam_src/vishalam/train/muon.py'), exist_ok=True)
open('/kaggle/working/vishalam_src/vishalam/train/muon.py', 'w').write('"""Muon optimizer (MomentUm Orthogonalized by Newton-schulz), written from scratch.\n\nIdea (Keller Jordan, modded-nanogpt): for a 2-D weight matrix, take the momentum-smoothed gradient\nand replace it by the nearest (semi-)orthogonal matrix, i.e. keep its "directions" but set all\nsingular values to ~1. That update is computed with a few Newton-Schulz iterations. It is used only\nfor the hidden 2-D matrices of the transformer; embeddings, the (tied) output head and all 1-D\nparameters (norms) keep using AdamW.\n"""\nfrom __future__ import annotations\n\nimport torch\n\nNS_COEFFS = (3.4445, -4.7750, 2.0315)  # quintic iteration tuned to converge fast from a normalized start\n\n\ndef newton_schulz(G: torch.Tensor, steps: int = 5, eps: float = 1e-7) -> torch.Tensor:\n    """Approximately orthogonalize a 2-D matrix: returns ~U @ V^T for G = U S V^T.\n    Runs in fp32 (T4 GPUs have no fast bf16; the cost is tiny next to the forward/backward)."""\n    a, b, c = NS_COEFFS\n    X = G.float()\n    transposed = X.size(0) > X.size(1)\n    if transposed:\n        X = X.T\n    X = X / (X.norm() + eps)  # spectral norm <= Frobenius norm <= 1, so the iteration converges\n    for _ in range(steps):\n        A = X @ X.T\n        B = b * A + c * (A @ A)\n        X = a * X + B @ X\n    return X.T if transposed else X\n\n\nclass Muon(torch.optim.Optimizer):\n    """Muon for 2-D parameters.\n\n    Param-group options: lr, momentum, nesterov, weight_decay (decoupled), ns_steps, and\n    split (orthogonalize a fused matrix in `split` equal row-blocks, e.g. 3 for a fused q/k/v).\n    """\n\n    def __init__(self, params, lr: float = 0.02, momentum: float = 0.95, nesterov: bool = True,\n                 weight_decay: float = 0.0, ns_steps: int = 5, split: int = 1):\n        defaults = dict(lr=lr, momentum=momentum, nesterov=nesterov, weight_decay=weight_decay,\n                        ns_steps=ns_steps, split=split)\n        super().__init__(params, defaults)\n        for g in self.param_groups:\n            for p in g["params"]:\n                if p.dim() != 2:\n                    raise ValueError("Muon only handles 2-D parameters; use AdamW for the rest")\n\n    @torch.no_grad()\n    def step(self, closure=None):\n        loss = None\n        if closure is not None:\n            with torch.enable_grad():\n                loss = closure()\n        for g in self.param_groups:\n            lr, mom, split = g["lr"], g["momentum"], g["split"]\n            for p in g["params"]:\n                if p.grad is None:\n                    continue\n                grad = p.grad\n                state = self.state[p]\n                if "momentum_buffer" not in state:\n                    state["momentum_buffer"] = torch.zeros_like(grad)\n                buf = state["momentum_buffer"]\n                buf.lerp_(grad, 1 - mom)                          # buf = mom*buf + (1-mom)*grad\n                update = grad.lerp(buf, mom) if g["nesterov"] else buf\n                if split > 1:\n                    parts = update.chunk(split, dim=0)\n                    ortho = torch.cat([newton_schulz(u, g["ns_steps"]) for u in parts], dim=0)\n                    rows = p.size(0) // split\n                else:\n                    ortho = newton_schulz(update, g["ns_steps"])\n                    rows = p.size(0)\n                scale = max(1.0, rows / p.size(1)) ** 0.5         # keep update RMS comparable across shapes\n                if g["weight_decay"]:\n                    p.mul_(1 - lr * g["weight_decay"])\n                p.add_(ortho.to(p.dtype), alpha=-lr * scale)\n        return loss\n')
BENCH_PY = '\nimport os, statistics, sys, time\n# Kaggle\'s TPU VM sets this for a multi-host slice; torch_xla then expects 8 hosts ("Expected 8 worker\n# addresses, got 1") instead of 8 chips on one host.\nos.environ.pop("TPU_PROCESS_ADDRESSES", None)\nsys.path.insert(0, "/kaggle/working/vishalam_src")\nimport torch\nimport torch_xla\nimport torch_xla.core.xla_model as xm\nimport torch_xla.runtime as xr\nimport torch_xla.utils.checkpoint as xla_ckpt\nimport vishalam.model.model as vm\nfrom vishalam.model.config import ModelConfig\nfrom vishalam.model.model import Vishalam\nfrom vishalam.train.muon import Muon\n\nSIZES = {\n    "125m": dict(vocab_size=50304, d_model=768, n_layers=12, n_heads=12, ffn_dim=2048, max_seq_len=4096),\n    "350m": dict(vocab_size=50304, d_model=1024, n_layers=24, n_heads=16, ffn_dim=2816, max_seq_len=4096),\n}\n\n\ndef use_flash_attention():\n    import torch.nn.functional as TF\n    from torch_xla.experimental.custom_kernel import flash_attention\n\n    class _F:\n        def __getattr__(self, n):\n            return getattr(TF, n)\n\n        def scaled_dot_product_attention(self, q, k, v, attn_mask=None, is_causal=False, **kw):\n            assert attn_mask is None\n            return flash_attention(q, k, v, causal=is_causal, sm_scale=q.shape[-1] ** -0.5)\n\n    vm.F = _F()\n\n\ndef run(index, size, micro, ckpt, flash, steps=12, warm=4, seq=4096):\n    vm.checkpoint = lambda fn, *a, use_reentrant=False: xla_ckpt.checkpoint(fn, *a)\n    if flash:\n        use_flash_attention()\n    dev = torch_xla.device()\n    world, rank = xr.world_size(), xr.global_ordinal()\n    torch.manual_seed(0)\n    model = Vishalam(ModelConfig(**SIZES[size])).to(dev)\n    model.grad_checkpointing = ckpt\n    hidden = [p for n, p in model.named_parameters() if n.startswith("blocks.") and p.dim() == 2]\n    other = [p for n, p in model.named_parameters() if not (n.startswith("blocks.") and p.dim() == 2)]\n    opts = [Muon(hidden, lr=0.02), torch.optim.AdamW(other, lr=3e-4, betas=(0.9, 0.95), foreach=True)]\n    params = [p for p in model.parameters()]\n    g = torch.Generator().manual_seed(rank)\n    x = torch.randint(0, 50304, (micro, seq + 1), generator=g).to(dev)\n    times = []\n    for i in range(steps):\n        t = time.time()\n        with torch.autocast("xla", dtype=torch.bfloat16):\n            _, loss = model(x[:, :-1], x[:, 1:])\n        loss.backward()\n        xm.all_reduce(xm.REDUCE_SUM, [p.grad for p in params], scale=1.0 / world)\n        for o in opts:\n            o.step()\n            o.zero_grad(set_to_none=False)\n        xm.mark_step()\n        xm.wait_device_ops()\n        times.append(time.time() - t)\n        if rank == 0:\n            print(f"  step {i}: {times[-1]:.2f}s loss {loss.item():.3f}", flush=True)\n    if rank == 0:\n        dt = statistics.median(times[warm:])\n        tps = micro * seq * world / dt\n        print(f"RESULT {size} chips={world} micro={micro}/chip ckpt={ckpt} flash={flash}: {tps:,.0f} tokens/s "\n              f"({dt:.2f}s/step, first step {times[0]:.0f}s) -> 10.5B tokens = {10.5e9 / tps / 3600:,.0f} h, "\n              f"6B = {6e9 / tps / 3600:,.0f} h", flush=True)\n\n\nif __name__ == "__main__":\n    size, micro, ckpt, flash = sys.argv[1], int(sys.argv[2]), sys.argv[3] == "1", sys.argv[4] == "1"\n    torch_xla.launch(run, args=(size, micro, ckpt, flash))\n'
print("source written")


In [ ]:
# --- 3. Run each config in its own process (an out-of-memory config only loses itself) ---
import subprocess, sys
open("/kaggle/working/bench.py", "w").write(BENCH_PY)
CONFIGS = [  # size, micro per chip, activation checkpointing, flash attention
    ("350m", 4, 1, 0),
    ("350m", 4, 1, 1),
    ("350m", 8, 1, 1),
    ("350m", 4, 0, 1),
]
summary = []
for size, micro, ckpt, flash in CONFIGS:
    print(f"==== {size} micro={micro} ckpt={ckpt} flash={flash}", flush=True)
    try:
        r = subprocess.run([sys.executable, "/kaggle/working/bench.py", size, str(micro), str(ckpt), str(flash)],
                           capture_output=True, text=True, timeout=1500)
        out = r.stdout + r.stderr
    except subprocess.TimeoutExpired as e:
        out = f"TIMEOUT {e}"
    lines = [l for l in out.splitlines() if "RESULT" in l or "step " in l or "Error" in l or "error" in l]
    print("\n".join(lines[-20:]) or out[-3000:], flush=True)
    summary += [l for l in out.splitlines() if l.startswith("RESULT")] or [f"FAILED {size} micro={micro} ckpt={ckpt} flash={flash}"]
print("\n==== SUMMARY ====")
print("\n".join(summary))
